# NLP Code Along

#### Task
Build an SMS spam classifier using Spark for data handling and sklearn for feature extraction and modeling.

Dataset: UCI SMS Spam Collection.
https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection

#### Import Spark and initialize session.

In [0]:
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName('nlp').getOrCreate()

#### Load raw SMS dataset and rename columns.

In [0]:
data = spark.read.csv("data/sms_spam_collection",inferSchema=True,sep='\t')

In [0]:
data = data.withColumnRenamed('_c0','class').withColumnRenamed('_c1','text')

In [0]:
data.show()

+-----+--------------------+
|class|                text|
+-----+--------------------+
|  ham|Go until jurong p...|
|  ham|Ok lar... Joking ...|
| spam|Free entry in 2 a...|
|  ham|U dun say so earl...|
|  ham|Nah I don't think...|
| spam|FreeMsg Hey there...|
|  ham|Even my brother i...|
|  ham|As per your reque...|
| spam|WINNER!! As a val...|
| spam|Had your mobile 1...|
|  ham|I'm gonna be home...|
| spam|SIX chances to wi...|
| spam|URGENT! You have ...|
|  ham|I've been searchi...|
|  ham|I HAVE A DATE ON ...|
| spam|XXXMobileMovieClu...|
|  ham|Oh k...i'm watchi...|
|  ham|Eh u remember how...|
|  ham|Fine if thats th...|
| spam|England v Macedon...|
+-----+--------------------+
only showing top 20 rows


#### Clean and prepare the data.

#### Create a message length feature as an additional signal.

In [0]:
from pyspark.sql.functions import length

In [0]:
data = data.withColumn('length',length(data['text']))

In [0]:
data.show()

+-----+--------------------+------+
|class|                text|length|
+-----+--------------------+------+
|  ham|Go until jurong p...|   111|
|  ham|Ok lar... Joking ...|    29|
| spam|Free entry in 2 a...|   155|
|  ham|U dun say so earl...|    49|
|  ham|Nah I don't think...|    61|
| spam|FreeMsg Hey there...|   147|
|  ham|Even my brother i...|    77|
|  ham|As per your reque...|   160|
| spam|WINNER!! As a val...|   157|
| spam|Had your mobile 1...|   154|
|  ham|I'm gonna be home...|   109|
| spam|SIX chances to wi...|   136|
| spam|URGENT! You have ...|   155|
|  ham|I've been searchi...|   196|
|  ham|I HAVE A DATE ON ...|    35|
| spam|XXXMobileMovieClu...|   149|
|  ham|Oh k...i'm watchi...|    26|
|  ham|Eh u remember how...|    81|
|  ham|Fine if thats th...|    56|
| spam|England v Macedon...|   155|
+-----+--------------------+------+
only showing top 20 rows


In [0]:
# Pretty Clear Difference
data.groupby('class').mean().show()

+-----+-----------------+
|class|      avg(length)|
+-----+-----------------+
| spam|138.6706827309237|
|  ham| 71.4545266210897|
+-----+-----------------+



#### Convert Spark data to pandas and set up sklearn NLP pipeline components.

In [0]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [0]:
nlp_pdf = data.select('class', 'text', 'length').toPandas()
nlp_pdf['label'] = nlp_pdf['class'].map({'ham': 0, 'spam': 1})

#### Create labels and split data for supervised training.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    nlp_pdf['text'],
    nlp_pdf['label'],
    test_size=0.3,
    random_state=42,
    stratify=nlp_pdf['label'],
)

### The Model

We'll use Naive Bayes, but feel free to play around with this choice!

In [0]:
spam_pipeline = Pipeline([
    ('count', CountVectorizer(stop_words='english')),
    ('tfidf', TfidfTransformer()),
    ('nb', MultinomialNB()),
])

In [0]:
spam_predictor = spam_pipeline.fit(X_train, y_train)

### Pipeline

### Training and Evaluation!

In [0]:
test_pred = spam_predictor.predict(X_test)
print(test_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]


In [0]:
data.printSchema()

root
 |-- class: string (nullable = true)
 |-- text: string (nullable = true)
 |-- length: integer (nullable = true)



In [0]:
acc = accuracy_score(y_test, test_pred)
print('Accuracy of model at predicting spam was: {}'.format(acc))

Accuracy of model at predicting spam was: 0.9731022115959355
